# TfR1 Challenge 04 — Proteina-Complexa binder designTarget: human TfR1 apical domain, **EP2 / His300** epitope(`A263 A264 A266 A269 A300 A303`), binder length 55-90.**Runtime: A100 or H100.** Runtime -> Change runtime type -> A100.Run cells in order. Each one prints a clear PASS/STOP line; do not continue past a STOP.Everything slow (checkpoints, designs) is mirrored to Google Drive at`/content/drive/MyDrive/tfr1_challenge04`. If the runtime drops, re-run from the top: the cells detect what isalready in Drive and restore instead of re-downloading.**Known-good configuration baked in** (all of these were failure modes during setup):- `uv pip install` is forced to target the venv interpreter, not `/usr`- `--system` is stripped from the build script- `llvmlite`/`numba` are pre-installed ahead of `tmol`- `complexa init` runs twice: once to create `.env`, once (`init uv`) to write `env.sh`- `.env` paths are rewritten to real Colab paths before `init uv`- every shell cell sources **both** `.venv/bin/activate` and `env.sh`- only `--complexa-all` is downloaded, never `--all` (that pulls ESM2/AF2/RF3)

In [ ]:
# CELL 1 - preflight. STOP if this fails.
import subprocess, sys, shutil

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU :', gpu or 'NONE')
assert gpu, 'STOP: no GPU. Runtime -> Change runtime type -> A100.'
assert 'A100' in gpu or 'H100' in gpu, f'STOP: got "{gpu}". Needs A100/H100.'

rel = open('/etc/os-release').read()
ver = [l for l in rel.splitlines() if l.startswith('VERSION_ID=')][0].split('"')[1]
print('OS  :', ver)
assert float(ver) >= 22.04, f'STOP: Ubuntu {ver}; build needs 22.04+.'

free = shutil.disk_usage('/content').free / 1e9
print(f'Disk: {free:.0f} GB free')
assert free > 40, 'STOP: under 40 GB free.'

print('\nPASS - preflight')

In [ ]:
# CELL 2 - mount Drive. Everything slow gets mirrored here.
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/tfr1_challenge04'
for sub in ['', '/checkpoints', '/targets', '/outputs']:
    os.makedirs(DRIVE + sub, exist_ok=True)

print('Drive ready:', DRIVE)
for sub in ['checkpoints', 'targets', 'outputs']:
    n = len(os.listdir(f'{DRIVE}/{sub}'))
    print(f'  {sub}: {n} file(s) already saved')
print('\nPASS - drive')

In [ ]:
# CELL 3 - targets. Restores from Drive if already uploaded once.
import os, shutil
DRIVE = '/content/drive/MyDrive/tfr1_challenge04'
NEEDED = ['human_TfR1_apical.pdb', 'mouse_TfR1_apical.pdb']
os.makedirs('/content/targets', exist_ok=True)

restored = []
for f in NEEDED:
    src = f'{DRIVE}/targets/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/targets/{f}')
        restored.append(f)

missing = [f for f in NEEDED if not os.path.exists(f'/content/targets/{f}')]
if missing:
    print('Upload:', ', '.join(missing))
    from google.colab import files
    up = files.upload()
    for name in up:
        shutil.move(f'/content/{name}', f'/content/targets/{name}')

for f in NEEDED:
    p = f'/content/targets/{f}'
    if os.path.exists(p):
        shutil.copy(p, f'{DRIVE}/targets/{f}')

print('restored from Drive:', restored or 'none')
print('present:', sorted(os.listdir('/content/targets')))
print('\nPASS - targets')

In [ ]:
# CELL 4 - validate targets. STOP if this fails.
# A column-misaligned PDB looks fine in an editor and kills a run minutes in.
STD = set('ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL'.split())
HOT = [263, 264, 266, 269, 300, 303]
EXPECT = {263:'THR', 264:'PHE', 266:'GLU', 269:'ALA', 300:'HIS', 303:'LEU'}

for f in ['human_TfR1_apical.pdb', 'mouse_TfR1_apical.pdb']:
    path = f'/content/targets/{f}'
    nums, alt, resn, seen, ch = [], set(), set(), {}, set()
    for line in open(path):
        if line.startswith('ATOM'):
            n = int(line[22:26])
            nums.append(n); alt.add(line[16]); resn.add(line[17:20].strip()); ch.add(line[21])
            if n in HOT:
                seen[n] = line[17:20].strip()
    u = sorted(set(nums)); bad = sorted(resn - STD)
    gaps = [(u[i], u[i+1]) for i in range(len(u)-1) if u[i+1]-u[i] > 1]
    print(f'{f}: res {u[0]}-{u[-1]} n={len(u)} chains={sorted(ch)} altloc={sorted(alt)}')
    print(f'   non-standard: {bad or "none"}   gaps: {gaps or "none"}')
    print(f'   hotspots: {[(k, seen.get(k,"MISSING")) for k in HOT]}')
    assert not bad, f'STOP: {f} malformed (ATOM cols 18-20)'
    assert ch == {'A'}, f'STOP: {f} chains {sorted(ch)}, expected A'
    assert seen == EXPECT, f'STOP: {f} hotspot identities wrong'

print('\nThe mouse 209-211 gap is the real V210 deletion, not damage.')
print('PASS - targets valid, hotspots identical across species')

In [ ]:
# CELL 5 - clone and patch the build script. Fast.
import subprocess, os
from pathlib import Path

if not os.path.isdir('/content/Proteina-Complexa'):
    for org in ['NVIDIA-BioNeMo', 'NVIDIA-Digital-Bio']:
        r = subprocess.run(['git','clone',f'https://github.com/{org}/Proteina-Complexa.git','/content/Proteina-Complexa'],
                           capture_output=True, text=True)
        print(org, '->', 'ok' if r.returncode == 0 else r.stderr.strip()[:160])
        if r.returncode == 0:
            break
assert os.path.isdir('/content/Proteina-Complexa'), 'STOP: clone failed'

p = Path('/content/Proteina-Complexa/env/build_uv_env.sh')
t = p.read_text()
V = '/content/Proteina-Complexa/.venv/bin/python'

# Fix 1: --system makes uv ignore the venv it just built and install into the
#        host Python (3.13 on Colab). Everything else cascades from this.
# Fix 2: pin uv explicitly at the venv interpreter.
# Fix 3: pre-install llvmlite/numba so tmol's resolver cannot pick broken versions.
# Fix 4: atomworks is installed unversioned behind a `|| echo "Warning:"`, so a
#        wrong version installs silently. 3.x dropped AtomSelectionStack, which
#        pdb_utils.py imports -> generate dies on ImportError. 2.2.1 has it.
if f'--python {V}' not in t:
    t = t.replace('uv pip install', f'uv pip install --python {V}').replace(' --system', '')
    t = t.replace('"atomworks[ml,openbabel,dev]"', '"atomworks[ml,openbabel,dev]==2.2.1"')
    L = t.splitlines()
    i = next((i for i, l in enumerate(L) if 'tmol' in l and not l.strip().startswith('#')), None)
    if i is not None and 'llvmlite' not in t:
        L.insert(i, f'uv pip install --python {V} "llvmlite>=0.41" "numba>=0.59"')
    p.write_text('\n'.join(L) + '\n')
    print('patched')
else:
    print('already patched')

pinned = 'atomworks[ml,openbabel,dev]==2.2.1' in p.read_text()
print('atomworks pinned to 2.2.1:', pinned)
if not pinned:
    print('  WARNING: the atomworks install line did not match the expected text.')
    print('  Check it by hand -- unversioned atomworks installs 3.x and breaks generate.')
p.chmod(0o755)

print(subprocess.run(['bash','-lc',f'grep -c -- "--python {V}" {p}'],
                     capture_output=True, text=True).stdout.strip(), 'install lines pinned to venv')
print('\nPASS - patched')

In [ ]:
%%bash
# CELL 6 - build the environment. SLOW (~10 min). Watch the "Using Python" lines:
# they must point at .venv, never /usr.
cd /content/Proteina-Complexa
unset UV_SYSTEM_PYTHON UV_CONSTRAINT UV_NO_BUILD_PACKAGE
rm -rf .venv
./env/build_uv_env.sh 2>&1 | tee /content/build.log | grep -E "^\[|error|Using Python|Installed [0-9]"

In [ ]:
%%bash
# CELL 7 - verify the build. STOP if any line is wrong.
set +e
cd /content/Proteina-Complexa
source .venv/bin/activate
echo "packages: $(ls .venv/lib/python3.12/site-packages | wc -l)   (expect ~500+)"
echo "complexa: $(which complexa)"
python -c "import torch; print('torch   :', torch.__version__, '| cuda', torch.cuda.is_available())"
python -c "import tree; print('dm-tree : ok')"
python -c "import atomworks; print('atomworks:', atomworks.__version__)" 2>&1 | tail -1
python -c "from atomworks.io.utils.selection import AtomSelectionStack; print('AtomSelectionStack: ok')" 2>&1 | tail -1

In [ ]:
%%bash
# CELL 8 - configure .env and generate env.sh.
#
# complexa init is TWO phases:
#   complexa init        -> creates .env from .env_example
#   complexa init uv     -> writes env.sh (only once .env has real paths)
# The shipped .env has LOCAL_CODE_PATH=/path/to/Proteina-Complexa, a literal
# placeholder, and almost every other path derives from it.
set +e
cd /content/Proteina-Complexa
source .venv/bin/activate
mkdir -p /content/PFM_data

[ -f .env ] || complexa init < /dev/null 2>&1 | tail -3

sed -i \
  -e 's|^LOCAL_CODE_PATH=.*|LOCAL_CODE_PATH=/content/Proteina-Complexa|' \
  -e 's|^LOCAL_DATA_PATH=.*|LOCAL_DATA_PATH=/content/PFM_data|' \
  -e 's|^GITLAB_TOKEN=.*|GITLAB_TOKEN=|' \
  -e 's|^WANDB_API_KEY=.*|WANDB_API_KEY=|' \
  -e 's|^WANDB_ENTITY=.*|WANDB_ENTITY=|' \
  -e 's|^HF_TOKEN=.*|HF_TOKEN=|' \
  -e 's|^CLUSTER_USER=.*|CLUSTER_USER=colab|' .env
grep -q '^WANDB_MODE=' .env || echo 'WANDB_MODE=disabled' >> .env

grep -E '^(LOCAL_CODE_PATH|LOCAL_DATA_PATH)=' .env
complexa init uv --force 2>&1 | tail -4
source env.sh && echo "PASS - env.sh written and sourced"

In [ ]:
%%bash
# CELL 9 - checkpoints. Restores from Drive if present, otherwise downloads,
# then mirrors back to Drive so a runtime reset never costs this twice.
#
# --complexa-all only. NOT --all: that additionally pulls ESM2/AF2/RF3, which
# is what was still downloading when an earlier runtime died.
set +e
cd /content/Proteina-Complexa
source .venv/bin/activate
source env.sh
mkdir -p checkpoints

n=$(ls /content/drive/MyDrive/tfr1_challenge04/checkpoints/*.ckpt 2>/dev/null | wc -l)
if [ "$n" -gt 0 ]; then
  echo "restoring $n checkpoint(s) from Drive"
  cp -nv /content/drive/MyDrive/tfr1_challenge04/checkpoints/*.ckpt checkpoints/
else
  echo "downloading (slow, no progress output until done)"
  complexa download --complexa-all < /dev/null 2>&1 | tail -15
  find . -name '*.ckpt' -exec cp -nv {} /content/drive/MyDrive/tfr1_challenge04/checkpoints/ \;
fi

echo "=== on disk ==="
find . -name '*.ckpt' -exec ls -lh {} \; 2>/dev/null
echo "=== in Drive ==="
ls -lh /content/drive/MyDrive/tfr1_challenge04/checkpoints/ 2>/dev/null

In [ ]:
%%bash
# CELL 10 - register the target.
#   --target-input      chain A, residues 189-383 (apical domain)
#   --hotspot-residues  EP2 / His300 patch, all six identical in human and mouse
#   --binder-length     55 90 -> the 40-100 protein minibinder category
# User-added targets keep the name you give them, so the task name is "tfr1"
# (the numeric prefixes you see belong to the shipped targets).
set +e
cd /content/Proteina-Complexa
source .venv/bin/activate
source env.sh

complexa target add tfr1 --force \
  --target-path /content/targets/human_TfR1_apical.pdb \
  --target-input A189-383 \
  --hotspot-residues A263 A264 A266 A269 A300 A303 \
  --binder-length 55 90 2>&1 | tail -15

echo "=== confirm ==="
complexa target list 2>&1 | grep -i tfr1

In [ ]:
%%bash
# CELL 11 - resolve the config before spending GPU time. STOP if this fails.
set +e
cd /content/Proteina-Complexa
source .venv/bin/activate
source env.sh
complexa validate design configs/search_binder_local_pipeline.yaml \
  ++run_name=tfr1_ep2_his300 ++generation.task_name=tfr1 2>&1 | tail -25

In [ ]:
%%bash
# CELL 12 - GENERATE. This is the cell that produces designs.
#
# Running `generate` alone rather than the full `design` pipeline on purpose:
# design = generate -> filter -> evaluate -> analyze, and `evaluate` needs the
# AF2 community models we deliberately did not download. Generation co-designs
# backbone AND sequence, so this alone yields submittable designs. Filter and
# evaluate are optional extras below.
#
# gen_njobs defaults to 1, which is correct for a single Colab GPU.
cd /content/Proteina-Complexa
source .venv/bin/activate
source env.sh
complexa generate configs/search_binder_local_pipeline.yaml \
  ++run_name=tfr1_ep2_his300 ++generation.task_name=tfr1 2>&1 | tee /content/generate.log

In [ ]:
# CELL 13 - SAVE. Run this the moment cell 12 produces anything,
# even if it errored partway. Partial designs beat none.
import glob, os, shutil
DRIVE = '/content/drive/MyDrive/tfr1_challenge04'
RUN = 'tfr1_ep2_his300'

found = []
for ext in ('pdb','cif','csv','fasta','json','yaml'):
    for p in glob.glob(f'/content/Proteina-Complexa/**/*.{ext}', recursive=True):
        if RUN in p:
            found.append(p)

print(f'{len(found)} output file(s) for run "{RUN}"')
for p in found[:25]:
    print('  ', p.replace('/content/Proteina-Complexa/', ''))

for p in found:
    if os.path.isfile(p):
        flat = os.path.relpath(p, '/content/Proteina-Complexa').replace('/', '__')
        shutil.copy(p, f'{DRIVE}/outputs/{flat}')

for lg in ['/content/generate.log', '/content/build.log']:
    if os.path.exists(lg):
        shutil.copy(lg, f'{DRIVE}/outputs/')

print(f'\nmirrored to {DRIVE}/outputs  ({len(os.listdir(DRIVE + "/outputs"))} files)')

In [ ]:
%%bash
# CELL 14 (optional) - filter. Uses the model's own metrics; may work without AF2.
# Safe to skip. Re-run CELL 13 afterwards to save anything new.
set +e
cd /content/Proteina-Complexa
source .venv/bin/activate
source env.sh
complexa filter configs/search_binder_local_pipeline.yaml \
  ++run_name=tfr1_ep2_his300 ++generation.task_name=tfr1 2>&1 | tail -30

In [ ]:
# CELL 15 - package everything for handover.
import shutil, os
DRIVE = '/content/drive/MyDrive/tfr1_challenge04'
shutil.make_archive('/content/tfr1_designs', 'zip', f'{DRIVE}/outputs')
size = os.path.getsize('/content/tfr1_designs.zip') / 1e6
print(f'tfr1_designs.zip  {size:.1f} MB')
shutil.copy('/content/tfr1_designs.zip', f'{DRIVE}/tfr1_designs.zip')
from google.colab import files
files.download('/content/tfr1_designs.zip')

## If a cell failsDo not restart from cell 1 unless the runtime itself reset. Checkpoints andtargets are in Drive, so even a full reset only costs the ~10 minute env build.| Cell | If it fails ||---|---|| 6 | Check `Using Python` in the output. If it says `/usr`, cell 5's patch did not apply - re-run cell 5. || 7 | 0 packages means the build aborted; read `/content/build.log`. || 8 | If `env.sh` is still missing, `cat .env` and confirm `LOCAL_CODE_PATH` is a real path. || 9 | If the download errors on HuggingFace auth, put a token in `.env` as `HF_TOKEN=` and re-run cell 8 then 9. || 11 | Config did not resolve. Check `complexa target list` still shows `tfr1`. || 12 | Run cell 13 regardless, then send me `/content/generate.log`. |Send me `tfr1_designs.zip` and I will do the mouse cross-reactivity pass andbuild `submission.csv`. Challenge deadline: **Mon 2 Nov, 12:59 Paris**.